# Customer Segmentation 
In this notebook, we will segment our customer base using the K-Means clustering algorithm. We will use the RFM (Recency, Frequency, Monetary) metrics generated during the data cleaning phase to identify distinct customer personas, such as High-Value Customers or Occasional Buyers.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pickle

In [2]:
#load the rfm dataset created in notebook 01
rfm_df = pd.read_csv('../data/processed/customer_rfm_segments.csv')

print(f"loaded {len(rfm_df)} customer records for segmentation.")
display(rfm_df.head())

loaded 500 customer records for segmentation.


,user_id,recency,frequency,monetary
0,1,55,8,379242.93
1,2,25,10,391360.77
2,3,25,8,572880.44
3,4,149,7,454034.22
4,5,49,10,815819.12


## 1. Data Preprocessing (Scaling)
K-Means clustering is a distance-based algorithm. Because our RFM features have completely different scales (e.g., Recency is in days, Monetary is in thousands of dollars), we must standardize the data so that no single feature dominates the clustering process.

In [3]:
#isolate the features for clustering
features = ['recency', 'frequency', 'monetary']
X = rfm_df[features]

#initialize the standard scaler
scaler = StandardScaler()

#fit and transform the data
X_scaled = scaler.fit_transform(X)

#convert back to dataframe for easier inspection
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print("standardized rfm features:")
display(X_scaled_df.head())

standardized rfm features:


,recency,frequency,monetary
0,-0.311484,-0.626839,-1.028846
1,-0.666844,0.000000,-0.979173
2,-0.666844,-0.626839,-0.235094
3,0.801975,-0.940259,-0.722264
4,-0.382556,0.000000,0.760752


## 2. Applying K-Means Clustering
Based on the business scenario, we want to group our customers into 3 distinct segments. We will initialize a K-Means model with `n_clusters=3` and fit it to our scaled data.

In [4]:
#initialize kmeans with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

#fit the model and predict cluster labels
rfm_df['cluster'] = kmeans.fit_predict(X_scaled)

print("cluster assignments added to dataframe.")
display(rfm_df.head())

cluster assignments added to dataframe.


,user_id,recency,frequency,monetary,cluster
0,1,55,8,379242.93,2
1,2,25,10,391360.77,2
2,3,25,8,572880.44,2
3,4,149,7,454034.22,0
4,5,49,10,815819.12,1


## 3. Analyzing and Profiling the Segments
To assign meaningful business names to these clusters (like "High-value customers"), we need to analyze the average Recency, Frequency, and Monetary values for each cluster. 
* **High Monetary / High Frequency / Low Recency** -> High-value customers
* **Moderate Monetary / Moderate Frequency** -> Occasional buyers
* **Low Monetary / High Recency (haven't bought recently)** -> At-risk or Discount seekers

In [5]:
#calculate the mean rfm values for each cluster
cluster_summary = rfm_df.groupby('cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': ['mean', 'count']
}).reset_index()

cluster_summary.columns = ['cluster', 'avg_recency_days', 'avg_frequency', 'avg_monetary', 'customer_count']

#sort by monetary value to help assign labels programmatically
cluster_summary = cluster_summary.sort_values(by='avg_monetary', ascending=False).reset_index(drop=True)

#assign business labels based on monetary value ranking
label_mapping = {
    cluster_summary.loc[0, 'cluster']: 'Segment 1: High-value customers',
    cluster_summary.loc[1, 'cluster']: 'Segment 2: Occasional buyers',
    cluster_summary.loc[2, 'cluster']: 'Segment 3: Discount seekers / At-risk'
}

#map the labels back to the main dataframe
rfm_df['customer_segment'] = rfm_df['cluster'].map(label_mapping)
cluster_summary['segment_name'] = cluster_summary['cluster'].map(label_mapping)

print("customer segment profiles:")
display(cluster_summary[['segment_name', 'avg_recency_days', 'avg_frequency', 'avg_monetary', 'customer_count']])

customer segment profiles:


,segment_name,avg_recency_days,avg_frequency,avg_monetary,customer_count
0,Segment 1: High-value customers,51.674847,13.398773,888458.820123,163
1,Segment 2: Occasional buyers,53.452471,8.730038,522391.187034,263
2,Segment 3: Discount seekers / At-risk,245.500000,7.027027,444707.797703,74


## 4. Finalizing Output
Let's view a sample of individual users and their assigned segments to ensure the output matches the requested business format.

In [6]:
#display sample of user segments as requested by the business scenario
sample_users = rfm_df[['user_id', 'customer_segment']].sample(5, random_state=42)

print("example output format:")
for index, row in sample_users.iterrows():
    print(f"User: U{row['user_id']}")
    print(f"Classification: {row['customer_segment']}\n")
    
#save the final labeled dataset for dashboard reporting
rfm_df.to_csv('../data/processed/final_customer_segments.csv', index=False)

example output format:
User: U362
Classification: Segment 2: Occasional buyers

User: U74
Classification: Segment 2: Occasional buyers

User: U375
Classification: Segment 2: Occasional buyers

User: U156
Classification: Segment 2: Occasional buyers

User: U105
Classification: Segment 2: Occasional buyers



## 5. Serializing the Model Artifacts
We will save both the StandardScaler and the trained KMeans model. In a production environment, when a new customer's data arrives, we will first scale their metrics using this exact scaler before passing it to the KMeans model to predict their segment.

In [7]:
#save the scaler
with open('../models/rfm_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

#save the kmeans model
with open('../models/kmeans_model.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

print("K-Means clustering model and StandardScaler successfully saved to the models/ directory.")

K-Means clustering model and StandardScaler successfully saved to the models/ directory.
